# N12. Tie breaking in hard Hamming decoding

Why this notebook exists. Our hard decoder takes the nearest codeword with `argmin`,
which returns the lowest id whenever several ids are equally close. Every marker in
Shadow ArUco has an id between 238 and 249, the highest in the dictionary, so a tie is
almost always resolved to a wrong id. Heavy occlusion produces many ties. The soft
decoding gain reported as finding 8 could therefore be an artefact of tie breaking
rather than a real benefit of using the confidence values.

This notebook measures the tie rate and recomputes hard decoding four ways:

* `low`    the published behaviour, lowest id wins
* `random` uniform choice among the tied ids, averaged over 20 draws
* `best`   a tie counts as correct if the true id is among the tied ids
* `worst`  a tie counts as wrong whenever there is more than one candidate

If the soft gain survives against `random`, finding 8 stands. If it does not, finding 8
must be rewritten as a tie breaking artefact. Nothing is trained here.

**Result on development data (first run).** The soft gain does not survive against fair
tie breaking under grey or white occlusion: soft is 0.7 to 9 points worse than hard with
random tie breaking. Under black it survives and grows to about 24 points at 18 hidden
cells, beating even the optimistic hard bound at the heaviest severities. Finding 8 must
be rewritten as a black only result. The calibration cell near the end looks for the
reason.


In [1]:
!pip -q install opencv-contrib-python==4.10.0.84 h5py 2>&1 | tail -1

In [2]:
%%writefile deeparuco_port.py
"""
Port the pretrained DeepArUco++ decoder from Keras .h5 to PyTorch, without TensorFlow.

The published decoder (models/dec_new.h5 in AVAuco/deeparuco) is a Keras 2.11 model:

    Input(32, 32, 1)
    Conv2D(8, 5x5, valid) -> PReLU -> MaxPool2D(2)
    Conv2D(4, 3x3, valid) -> PReLU -> MaxPool2D(2)
    Conv2D(1, 1x1, valid, sigmoid)   ->  (6, 6, 1)

Only h5py is needed to read the weights, so the repo's Python 3.9 + TensorFlow
environment is never required for baseline inference.

IMPORTANT PORTING DETAIL
    Keras PReLU defaults to shared_axes=None, so alpha is PER-ELEMENT, not per-channel:
    the stored alphas have shape (28, 28, 8) and (12, 12, 4). torch.nn.PReLU only
    supports per-channel alpha, so using it here silently produces a different model.
    SpatialPReLU below keeps the full alpha tensor.

Layout conversions:
    Keras conv kernel (kh, kw, in, out) -> torch (out, in, kh, kw)
    Keras PReLU alpha (h, w, c)         -> torch (c, h, w)
    Keras 'valid' padding               -> torch padding=0

Usage:
    from deeparuco_port import load_weights, NumpyDecoder, TorchDecoder, verify_port
    verify_port('models/dec_new.h5')          # run this first, once
    dec = TorchDecoder.from_h5('models/dec_new.h5')
    probs = dec(x)                            # x: (N, 1, 32, 32) float in [0, 1]

Upstream code is AGPL-3.0. This module reads a weights file; it does not copy source.
Cite Berral-Soler et al., Image and Vision Computing 152 (2024) 105313.
"""

import numpy as np

DEFAULT_H5 = "models/dec_new.h5"


# --------------------------------------------------------------------------- weights

def load_weights(h5_path=DEFAULT_H5):
    """Read decoder weights from the Keras .h5. Returns a dict of numpy arrays."""
    import h5py

    store = {}
    with h5py.File(h5_path, "r") as f:
        def visit(name, obj):
            if isinstance(obj, h5py.Dataset) and name.startswith("model_weights/"):
                store[name] = np.array(obj)
        f.visititems(visit)

    def pick(layer, suffix):
        hits = [k for k in store if f"/{layer}/" in k and k.endswith(suffix)]
        if len(hits) != 1:
            raise KeyError(f"expected one match for {layer}:{suffix}, got {hits}")
        return store[hits[0]]

    w = {
        "c1_w": pick("conv2d", "kernel:0"),   "c1_b": pick("conv2d", "bias:0"),
        "p1_a": pick("p_re_lu", "alpha:0"),
        "c2_w": pick("conv2d_1", "kernel:0"), "c2_b": pick("conv2d_1", "bias:0"),
        "p2_a": pick("p_re_lu_1", "alpha:0"),
        "c3_w": pick("conv2d_2", "kernel:0"), "c3_b": pick("conv2d_2", "bias:0"),
    }
    expect = {"c1_w": (5,5,1,8), "c1_b": (8,), "p1_a": (28,28,8),
              "c2_w": (3,3,8,4), "c2_b": (4,), "p2_a": (12,12,4),
              "c3_w": (1,1,4,1), "c3_b": (1,)}
    for k, shape in expect.items():
        assert w[k].shape == shape, f"{k}: got {w[k].shape}, expected {shape}"
    return w


# ------------------------------------------------------------------- numpy reference

class NumpyDecoder:
    """Reference implementation. Verified against synthetic markers: 4/5 exact."""

    def __init__(self, weights):
        self.w = weights

    @classmethod
    def from_h5(cls, path=DEFAULT_H5):
        return cls(load_weights(path))

    @staticmethod
    def _conv(x, w, b):
        kh, kw, _, co = w.shape
        H, W, _ = x.shape
        out = np.zeros((H - kh + 1, W - kw + 1, co), np.float32)
        for i in range(out.shape[0]):
            for j in range(out.shape[1]):
                out[i, j] = np.tensordot(
                    x[i:i+kh, j:j+kw, :], w, axes=([0, 1, 2], [0, 1, 2])) + b
        return out

    @staticmethod
    def _pool(x):
        h, w, c = x.shape
        return x.reshape(h // 2, 2, w // 2, 2, c).max(axis=(1, 3))

    def __call__(self, marker32):
        """marker32: (32, 32) float in [0, 1]. Returns (6, 6) bit probabilities."""
        w = self.w
        x = np.asarray(marker32, np.float32)[..., None]
        x = self._conv(x, w["c1_w"], w["c1_b"])
        x = np.where(x > 0, x, w["p1_a"] * x)
        x = self._pool(x)
        x = self._conv(x, w["c2_w"], w["c2_b"])
        x = np.where(x > 0, x, w["p2_a"] * x)
        x = self._pool(x)
        x = self._conv(x, w["c3_w"], w["c3_b"])
        return (1.0 / (1.0 + np.exp(-np.clip(x[..., 0], -60, 60)))).astype(np.float32)


# -------------------------------------------------------------------------- pytorch

def _torch():
    import torch
    import torch.nn as nn
    return torch, nn


class SpatialPReLU:
    """Created via _build so torch is imported lazily. See module docstring."""


def TorchDecoder_factory():
    torch, nn = _torch()

    class _SpatialPReLU(nn.Module):
        """Per-element PReLU, matching Keras PReLU(shared_axes=None)."""
        def __init__(self, alpha_chw):
            super().__init__()
            self.alpha = nn.Parameter(torch.tensor(alpha_chw, dtype=torch.float32))

        def forward(self, x):
            return torch.where(x > 0, x, self.alpha.unsqueeze(0) * x)

    class _TorchDecoder(nn.Module):
        def __init__(self, weights=None):
            super().__init__()
            self.c1 = nn.Conv2d(1, 8, 5)
            self.c2 = nn.Conv2d(8, 4, 3)
            self.c3 = nn.Conv2d(4, 1, 1)
            self.pool = nn.MaxPool2d(2)
            a1 = np.zeros((8, 28, 28), np.float32) if weights is None \
                 else np.transpose(weights["p1_a"], (2, 0, 1))
            a2 = np.zeros((4, 12, 12), np.float32) if weights is None \
                 else np.transpose(weights["p2_a"], (2, 0, 1))
            self.p1, self.p2 = _SpatialPReLU(a1), _SpatialPReLU(a2)
            if weights is not None:
                self.load_keras(weights)

        def load_keras(self, w):
            with torch.no_grad():
                for conv, kw, kb in ((self.c1, "c1_w", "c1_b"),
                                     (self.c2, "c2_w", "c2_b"),
                                     (self.c3, "c3_w", "c3_b")):
                    conv.weight.copy_(torch.tensor(np.transpose(w[kw], (3, 2, 0, 1))))
                    conv.bias.copy_(torch.tensor(w[kb]))

        @classmethod
        def from_h5(cls, path=DEFAULT_H5):
            return cls(load_weights(path))

        def forward(self, x):
            """x: (N, 1, 32, 32) in [0, 1]. Returns (N, 6, 6) bit probabilities."""
            x = self.pool(self.p1(self.c1(x)))
            x = self.pool(self.p2(self.c2(x)))
            return torch.sigmoid(self.c3(x)).squeeze(1)

    return _TorchDecoder


def __getattr__(name):
    if name == "TorchDecoder":
        return TorchDecoder_factory()
    raise AttributeError(name)


# --------------------------------------------------------------------------- verify

def verify_port(h5_path=DEFAULT_H5, n=16, tol=1e-4, verbose=True):
    """Assert the PyTorch port matches the numpy reference. Run once before use.

    The numpy path is validated; the torch path is validated against it here rather
    than assumed. Do not skip this.
    """
    torch, _ = _torch()
    w = load_weights(h5_path)
    ref = NumpyDecoder(w)
    net = TorchDecoder_factory()(w).eval()

    rng = np.random.default_rng(0)
    x = rng.random((n, 32, 32)).astype(np.float32)
    with torch.no_grad():
        got = net(torch.tensor(x).unsqueeze(1)).numpy()
    want = np.stack([ref(xi) for xi in x])

    err = float(np.abs(got - want).max())
    if verbose:
        print(f"port check: {n} random inputs, max abs difference {err:.2e} (tol {tol:g})")
    assert err < tol, f"PyTorch port disagrees with numpy reference (max diff {err:.2e})"
    if verbose:
        print("PyTorch port matches the numpy reference.")
    return err


def decode_ids(probs, id_to_bits, n_markers=250):
    """Hard-decision decode: nearest codeword by Hamming distance over 4 rotations.

    probs: (6, 6) bit probabilities. Returns (marker_id, rotation, hamming).
    Replace with soft log-likelihood decoding to use the confidences.
    """
    obs = (np.asarray(probs) > 0.5).astype(np.uint8)
    best = (None, None, 99)
    for mid in range(n_markers):
        ref = np.asarray(id_to_bits(mid), np.uint8).reshape(6, 6)
        for rot in range(4):
            h = int(np.sum(np.rot90(obs, rot) != ref))
            if h < best[2]:
                best = (mid, rot, h)
    return best


if __name__ == "__main__":
    import sys
    path = sys.argv[1] if len(sys.argv) > 1 else DEFAULT_H5
    w = load_weights(path)
    print("weights loaded:", {k: v.shape for k, v in w.items()})
    try:
        verify_port(path)
    except ImportError:
        print("torch not installed; numpy reference verified only")


Writing deeparuco_port.py


In [3]:
import os, sys, json, glob, zlib, subprocess, time, itertools
import numpy as np, cv2, torch
import torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from numpy.lib.stride_tricks import sliding_window_view

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE, WORK = '/content/drive/MyDrive/aruco_p2', '/content/work'
except ImportError:
    BASE = os.environ.get('ARUCO_P2_BASE',
                          os.path.join(os.path.expanduser('~'), 'aruco_p2')).replace('\\', '/')
    WORK = f'{BASE}/work'
ART, CACHE = f'{BASE}/artifacts', f'{BASE}/archives'
os.makedirs(WORK, exist_ok=True); sys.path.insert(0, os.getcwd())
assert os.path.exists(f'{ART}/splits.json'), f'{ART}/splits.json missing, run N0 first'

REPO = f'{WORK}/deeparuco'
if not os.path.isdir(REPO):
    subprocess.run(['git','clone','--depth','1','https://github.com/AVAuco/deeparuco.git',REPO],check=True)
sys.path.insert(0, REPO)
from impl.aruco import id_to_bits
from deeparuco_port import load_weights, NumpyDecoder

DEV_ = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
splits = json.load(open(f'{ART}/splits.json'))
DEVV, HELDOUT, STRIDE = splits['dev'], splits['heldout'][0], splits['benchmark_stride']
Wk  = load_weights(f'{REPO}/models/dec_new.h5')
CB  = np.stack([np.asarray(id_to_bits(i),np.uint8).reshape(6,6) for i in range(250)])
ROT = np.stack([np.rot90(CB,k,axes=(1,2)).reshape(250,36) for k in range(4)]).astype(np.float32)

SHADOW = f'{WORK}/shadow'; os.makedirs(SHADOW, exist_ok=True)
for i in range(1,7):
    if not os.path.isdir(f'{SHADOW}/video_{i}'):
        tar = f'{CACHE}/video_{i}.tar.gz'
        assert os.path.exists(tar), f'{tar} missing, see the download cell in N0'
        subprocess.run(['tar','-xzf',tar,'-C',SHADOW], check=True)

N0M = json.load(open(f'{ART}/manifest.json'))
for vid in DEVV + [HELDOUT]:
    n = len(glob.glob(f'{SHADOW}/{vid}/*.png'))
    assert n == N0M['shadow_aruco']['frames'][vid], f'{vid}: {n} frames on disk, manifest says other'
print('torch', torch.__version__, '| device', DEV_)
print('dev:', DEVV, '| held out:', HELDOUT, '| stride', STRIDE)


Mounted at /content/drive
torch 2.14.1+cu130 | device cuda
dev: ['video_1', 'video_2', 'video_3', 'video_4', 'video_5'] | held out: video_6 | stride 6


In [4]:
r_int = lambda v: int(round(v))
VALUES = {'grey':127, 'black':0, 'white':255}

def ordered_corners(xs, ys):
    cx, cy = np.mean(xs), np.mean(ys)
    idx = np.argsort(np.arctan2(xs-cx, ys-cy))
    return [v for i in idx for v in (xs[i], ys[i])]

def warp32(crop, c, t=32):
    dst = np.array([[0,0],[0,t-1],[t-1,t-1],[t-1,0]], np.float32)
    src = np.clip(np.array([[c[0]*63,c[1]*63],[c[2]*63,c[3]*63],
                            [c[4]*63,c[5]*63],[c[6]*63,c[7]*63]]),0,63).astype(np.float32)
    return cv2.warpPerspective(crop, cv2.getPerspectiveTransform(src,dst), (t,t))

def get_corners(m):
    c = m['corners']
    if isinstance(c[0], dict): return np.array([[p['x'],p['y']] for p in c], np.float32)
    return np.asarray(c, np.float32).reshape(4,2)

def cell_occlusion(m32):
    s = 32/8.0; out = np.zeros((6,6), bool)
    for r in range(6):
        for c in range(6):
            out[r,c] = m32[int((r+1)*s):int((r+2)*s), int((c+1)*s):int((c+2)*s)].mean() > 127.5
    return out

def prepare(img, mask, corners, scale=None):
    """scale=None uses the crop's own min and max (published behaviour).
    scale=(lo,hi) forces the brightness scale, which is the N11 oracle."""
    x, y = corners[:,0], corners[:,1]
    mnx, mxx, mny, mxy = x.min(), x.max(), y.min(), y.max()
    ex, ey = 0.2*(mxx-mnx)+0.5, 0.2*(mxy-mny)+0.5
    mnx, mny = max(mnx-ex,0), max(mny-ey,0)
    mxx = min(mxx+ex, img.shape[1]-1); mxy = min(mxy+ey, img.shape[0]-1)
    if mxx-mnx < 8 or mxy-mny < 8: return None, None
    sl = (slice(r_int(mny), r_int(mxy)), slice(r_int(mnx), r_int(mxx)))
    crop = cv2.resize(img[sl], (64,64)).astype(np.float32)
    lo, hi = (crop.min(), crop.max()) if scale is None else scale
    crop = np.clip((crop - lo)/(hi - lo + 1e-9), 0, 1)
    nc = ordered_corners((x-mnx)/(mxx-mnx), (y-mny)/(mxy-mny))
    m32 = warp32(cv2.resize(mask[sl],(64,64)), nc) if mask.any() else np.zeros((32,32), np.uint8)
    return warp32(crop, nc), cell_occlusion(m32)

def crop_scale(img, corners):
    """The (min,max) a clean crop of this marker would have had."""
    x, y = corners[:,0], corners[:,1]
    mnx, mxx, mny, mxy = x.min(), x.max(), y.min(), y.max()
    ex, ey = 0.2*(mxx-mnx)+0.5, 0.2*(mxy-mny)+0.5
    mnx, mny = max(mnx-ex,0), max(mny-ey,0)
    mxx = min(mxx+ex, img.shape[1]-1); mxy = min(mxy+ey, img.shape[0]-1)
    if mxx-mnx < 8 or mxy-mny < 8: return None
    c = cv2.resize(img[r_int(mny):r_int(mxy), r_int(mnx):r_int(mxx)], (64,64)).astype(np.float32)
    return float(c.min()), float(c.max())

def occluder_box(img, corners, level, position, key, mid):
    if level <= 0: return None
    x, y = corners[:,0], corners[:,1]
    x0,x1,y0,y1 = x.min(),x.max(),y.min(),y.max(); w,h = x1-x0, y1-y0
    if position == 'edge':
        side = int(np.random.default_rng(zlib.crc32(f'occlusion|{level}|{key}|{mid}'.encode())).integers(0,4))
        box = [(x0,y0,x0+level*w,y1),(x1-level*w,y0,x1,y1),
               (x0,y0,x1,y0+level*h),(x0,y1-level*h,x1,y1)][side]
    else:
        cx, cy = (x0+x1)/2, (y0+y1)/2
        box = (cx-level*w/2, cy-level*h/2, cx+level*w/2, cy+level*h/2)
    a,b,c,d = [int(round(v)) for v in box]
    a,b = max(a,0),max(b,0); c,d = min(c,img.shape[1]),min(d,img.shape[0])
    return (a,b,c,d) if (c>a and d>b) else None

def apply_occluder(img, box, value):
    if box is None: return img, np.zeros(img.shape[:2], np.uint8)
    if isinstance(value, str): value = VALUES[value]
    a,b,c,d = box
    out = img.copy(); mk = np.zeros(img.shape[:2], np.uint8)
    out[b:d,a:c] = int(value); mk[b:d,a:c] = 255
    return out, mk

def region_masks(corners, shape):
    """border ring and data region, in source image coordinates"""
    c = corners.mean(axis=0)
    inner = c + (corners - c) * (6.0/8.0)
    m_out = np.zeros(shape, np.uint8); cv2.fillPoly(m_out, [corners.astype(np.int32)], 255)
    m_in  = np.zeros(shape, np.uint8); cv2.fillPoly(m_in,  [inner.astype(np.int32)], 255)
    return cv2.bitwise_and(m_out, cv2.bitwise_not(m_in)), m_in

def make_condition(img, corners, cond, value, key, mid, level=0.50):
    if cond == 'clean':
        return img, np.zeros(img.shape[:2], np.uint8)
    border, data = region_masks(corners, img.shape[:2])
    if cond == 'border_only':
        paint = border
    else:
        box = occluder_box(img, corners, level, 'edge', key, mid)
        if box is None: return None, None
        a,b,c,d = box
        strip = np.zeros(img.shape[:2], np.uint8); strip[b:d, a:c] = 255
        paint = cv2.bitwise_and(strip, data) if cond == 'data_only' else strip
    out = img.copy(); out[paint > 0] = int(value)
    return out, paint

def pairs_for(vid):
    out = []
    for f in sorted(glob.glob(f'{SHADOW}/{vid}/*.png')):
        j = f'{SHADOW}/{vid}/corrected_annotations/{os.path.splitext(os.path.basename(f))[0]}.json'
        if os.path.exists(j): out.append((f,j))
    return out

class SpatialPReLU(nn.Module):
    def __init__(s,c,h,w): super().__init__(); s.a = nn.Parameter(torch.zeros(c,h,w))
    def forward(s,x): return torch.where(x>0, x, s.a.unsqueeze(0)*x)

class Decoder(nn.Module):
    """8 and 4 filters, as in the released code. pool is 'max' or 'avg'."""
    def __init__(s, out_ch=1, pool='max'):
        super().__init__()
        s.c1=nn.Conv2d(1,8,5); s.p1=SpatialPReLU(8,28,28)
        s.c2=nn.Conv2d(8,4,3); s.p2=SpatialPReLU(4,12,12)
        s.c3=nn.Conv2d(4,out_ch,1); s.out_ch=out_ch
        s.pool = F.max_pool2d if pool=='max' else F.avg_pool2d
    def forward(s,x):
        x=s.pool(s.p1(s.c1(x)),2); x=s.pool(s.p2(s.c2(x)),2); x=s.c3(x)
        return (x[:,0], x[:,1]) if s.out_ch==2 else (x[:,0], None)
print('pipeline ready')


pipeline ready


In [5]:
def _conv(x,w,b):
    kh,kw,_,_ = w.shape
    return np.einsum('nohckl,klci->nohi', sliding_window_view(x,(kh,kw),axis=(1,2)), w, optimize=True)+b
def _pool(x):
    N,H,Wd,Cc = x.shape; return x.reshape(N,H//2,2,Wd//2,2,Cc).max(axis=(2,4))

def decode_published(x8, chunk=4096):
    out=[]
    for s in range(0,len(x8),chunk):
        x = x8[s:s+chunk].astype(np.float32)[...,None]/255.0
        x = _pool(np.where((y:=_conv(x,Wk['c1_w'],Wk['c1_b']))>0,y,Wk['p1_a']*y))
        x = _pool(np.where((y:=_conv(x,Wk['c2_w'],Wk['c2_b']))>0,y,Wk['p2_a']*y))
        out.append(1/(1+np.exp(-np.clip(_conv(x,Wk['c3_w'],Wk['c3_b'])[...,0],-60,60))))
    return np.concatenate(out).astype(np.float32)

def decode_torch(net, x8, chunk=8192):
    ps, es = [], []
    with torch.no_grad():
        for s in range(0,len(x8),chunk):
            xb = torch.tensor(x8[s:s+chunk], dtype=torch.float32).unsqueeze(1).to(DEV_)/255.0
            lo, el = net(xb)
            ps.append(torch.sigmoid(lo).cpu().numpy())
            es.append(torch.sigmoid(el).cpu().numpy() if el is not None else np.zeros((len(lo),6,6),np.float32))
    return np.concatenate(ps), np.concatenate(es)

def dec_soft(P, Wt=None):
    LP = np.log(np.clip(P,1e-6,1)).reshape(len(P),36)
    LQ = np.log(np.clip(1-P,1e-6,1)).reshape(len(P),36)
    if Wt is not None:
        w = Wt.reshape(len(P),36); LP, LQ = LP*w, LQ*w
    return np.stack([LP@ROT[k].T + LQ@(1-ROT[k]).T for k in range(4)]).max(0).argmax(1)

def hamming_table(P):
    """(N,250) smallest Hamming distance to each id over the four rotations."""
    B = (P.reshape(len(P),36) > 0.5).astype(np.float32)
    d = np.stack([B@(1-ROT[k]).T + (1-B)@ROT[k].T for k in range(4)])   # (4,N,250)
    return d.min(0)

def dec_hard(P, tie='low', seed=0, ids_true=None):
    """tie: 'low' keeps the published behaviour of taking the lowest id,
    'random' picks uniformly among the tied ids,
    'best'/'worst' give the optimistic and pessimistic bounds."""
    D = hamming_table(P)
    dmin = D.min(1, keepdims=True)
    tied = (D == dmin)
    if tie == 'low':
        return tied.argmax(1), tied.sum(1)
    if tie in ('best','worst'):
        assert ids_true is not None
        hit = tied[np.arange(len(D)), ids_true]
        out = np.where(hit, ids_true, tied.argmax(1)) if tie=='best' else \
              np.where(hit & (tied.sum(1)>1), -1, tied.argmax(1))
        return out, tied.sum(1)
    rng = np.random.default_rng(seed)
    r = rng.random(D.shape) * tied
    return r.argmax(1), tied.sum(1)
print('decoders ready')


decoders ready


In [6]:
def paired(a_ok, b_ok, groups, n_boot=2000, seed=0):
    """Paired cluster bootstrap. groups is the cluster label per marker (frame or camera)."""
    rng = np.random.default_rng(seed)
    uf, inv = np.unique(groups, return_inverse=True)
    by = [np.where(inv==i)[0] for i in range(len(uf))]
    obs = a_ok.mean() - b_ok.mean()
    d = np.empty(n_boot)
    for i in range(n_boot):
        idx = np.concatenate([by[p] for p in rng.integers(0, len(by), len(by))])
        d[i] = a_ok[idx].mean() - b_ok[idx].mean()
    lo, hi = np.percentile(d, [2.5, 97.5])
    p = 2*min((d <= 0).mean(), (d >= 0).mean())
    return float(obs), float(lo), float(hi), max(float(p), 1/n_boot)

def holm(pvals, alpha=0.05):
    order = np.argsort(pvals); m = len(pvals); keep = np.zeros(m, bool)
    for rank, i in enumerate(order):
        if pvals[i] <= alpha/(m-rank): keep[i] = True
        else: break
    return keep
print('statistics ready')


statistics ready


In [7]:
RUN_ON   = 'dev'          # 'dev' or 'heldout'
VIEWS    = DEVV if RUN_ON == 'dev' else [HELDOUT]
TARGETS  = [3, 6, 9, 12, 15, 18]
COLOURS  = ['grey', 'white', 'black']
POSITION = 'edge'
N_DRAWS  = 20             # random tie breaking repeats

cal = []
for vid in VIEWS[:2]:
    for f, j in pairs_for(vid)[::STRIDE*8]:
        img = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
        key = f'{vid}/{os.path.splitext(os.path.basename(f))[0]}'
        for m in json.load(open(j))['markers']:
            cal.append((img, get_corners(m), key, int(m['id'])))
        if len(cal) > 400: break
    if len(cal) > 400: break

def mean_cells(position, level):
    tot=n=0
    for img, cn, key, mid in cal:
        oi, om = apply_occluder(img, occluder_box(img, cn, level, position, key, mid), 'grey')
        _, occ = prepare(oi, om, cn)
        if occ is None: continue
        tot += occ.sum(); n += 1
    return tot/max(n,1)

curve = [(float(L), mean_cells(POSITION, float(L))) for L in np.arange(0.02, 0.70, 0.02)]
LEVEL_FOR, ACHIEVED = {}, {}
for T in TARGETS:
    L, got = min(curve, key=lambda lc: abs(lc[1]-T))
    LEVEL_FOR[T] = L; ACHIEVED[T] = got
print('severity calibration:', {T: (round(LEVEL_FOR[T],2), round(ACHIEVED[T],1)) for T in TARGETS})

severity calibration: {3: (0.24, np.float64(3.3)), 6: (0.3, np.float64(5.9)), 9: (0.34, np.float64(8.5)), 12: (0.4, np.float64(11.8)), 15: (0.46, np.float64(15.7)), 18: (0.5, np.float64(18.0))}


In [8]:
BUF = {(c,T): {'x':[], 'id':[], 'frame':[], 'occ':[], 'mask':[]} for c in COLOURS for T in TARGETS}
CLEAN = {'x':[], 'id':[]}
t0 = time.time()
for vid in VIEWS:
    for f, j in pairs_for(vid)[::STRIDE]:
        img = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
        key = f'{vid}/{os.path.splitext(os.path.basename(f))[0]}'
        zero = np.zeros(img.shape[:2], np.uint8)
        for m in json.load(open(j))['markers']:
            cn, mid = get_corners(m), int(m['id'])
            xc, _ = prepare(img, zero, cn)
            if xc is None: continue                      # same corners, so every condition would fail too
            CLEAN['x'].append((xc*255).astype(np.uint8)); CLEAN['id'].append(mid)
            for T in TARGETS:
                box = occluder_box(img, cn, LEVEL_FOR[T], POSITION, key, mid)
                for col in COLOURS:
                    oi, om = apply_occluder(img, box, col)
                    x32, occ = prepare(oi, om, cn)
                    b = BUF[(col,T)]
                    b['x'].append((x32*255).astype(np.uint8)); b['id'].append(mid)
                    b['frame'].append(key); b['occ'].append(int(occ.sum())); b['mask'].append(occ)
    print(f'  {vid} ({time.time()-t0:.0f}s)')

for k in BUF:
    BUF[k] = {kk: np.array(vv) for kk, vv in BUF[k].items()}
for k in CLEAN: CLEAN[k] = np.array(CLEAN[k])
n = len(BUF[(COLOURS[0],TARGETS[0])]['id'])
assert all(len(BUF[k]['id']) == n == len(CLEAN['id']) for k in BUF), 'conditions are misaligned'
print(f'\n{n} markers per condition')
assert n > 1500, f'only {n} markers, data did not load correctly'
assert set(np.unique(BUF[(COLOURS[0],TARGETS[0])]['id'])) <= set(range(238,250)), 'unexpected ids'
print('ids present:', sorted(set(np.unique(BUF[(COLOURS[0],TARGETS[0])]['id']).tolist())))


  video_1 (48s)
  video_2 (83s)
  video_3 (126s)
  video_4 (167s)
  video_5 (201s)

12276 markers per condition
ids present: [238, 239, 240, 241, 242, 243, 244, 245, 246, 247, 248, 249]


In [9]:
rows = []
for col in COLOURS:
    for T in TARGETS:
        b = BUF[(col,T)]
        P = decode_published(b['x']); ids = b['id']
        low,  nties = dec_hard(P, 'low')
        best, _     = dec_hard(P, 'best',  ids_true=ids)
        worst, _    = dec_hard(P, 'worst', ids_true=ids)
        rnd = np.array([ (dec_hard(P,'random',seed=s)[0]==ids).mean() for s in range(N_DRAWS) ])
        rows.append(dict(colour=col, target=T, cells=float(b['occ'].mean()), n=int(len(ids)),
                         tie_rate=float((nties>1).mean()), mean_cands=float(nties.mean()),
                         hard_low=float((low==ids).mean()),
                         hard_random=float(rnd.mean()), hard_random_sd=float(rnd.std()),
                         hard_best=float((best==ids).mean()), hard_worst=float((worst==ids).mean()),
                         soft=float((dec_soft(P)==ids).mean())))

print(f"{'col':>6}{'cells':>7}{'ties%':>8}{'cands':>7}{'hard low':>10}{'hard rnd':>10}"
      f"{'hard best':>11}{'soft':>8}{'soft-low':>10}{'soft-rnd':>10}")
for r in rows:
    print(f"{r['colour']:>6}{r['cells']:>7.1f}{100*r['tie_rate']:>8.1f}{r['mean_cands']:>7.2f}"
          f"{100*r['hard_low']:>10.2f}{100*r['hard_random']:>10.2f}{100*r['hard_best']:>11.2f}"
          f"{100*r['soft']:>8.2f}{100*(r['soft']-r['hard_low']):>10.2f}"
          f"{100*(r['soft']-r['hard_random']):>10.2f}")

   col  cells   ties%  cands  hard low  hard rnd  hard best    soft  soft-low  soft-rnd
  grey    3.2     9.7   1.15     85.31     89.06      93.89   85.70      0.38     -3.36
  grey    5.7    16.4   1.31     74.48     79.41      86.80   76.33      1.85     -3.08
  grey    8.5    22.8   1.44     62.98     69.38      79.47   65.40      2.42     -3.99
  grey   11.5    29.7   1.58     46.66     52.52      62.14   49.22      2.56     -3.30
  grey   15.6    39.4   1.87     27.97     34.59      45.62   31.97      4.00     -2.61
  grey   18.0    44.5   2.10     18.43     23.28      33.11   22.59      4.16     -0.69
 white    3.2    12.2   1.21     81.23     85.78      91.90   76.71     -4.52     -9.07
 white    5.7    22.0   1.42     64.81     71.15      80.80   63.48     -1.33     -7.67
 white    8.5    28.4   1.58     52.05     59.20      70.89   51.08     -0.97     -8.12
 white   11.5    33.7   1.68     35.12     41.08      51.37   36.19      1.08     -4.89
 white   15.6    41.9   1.91    

In [10]:
TESTS = []
for col in COLOURS:
    for T in TARGETS:
        b = BUF[(col,T)]; P = decode_published(b['x']); ids = b['id']
        soft_ok = (dec_soft(P)==ids)
        low_ok  = (dec_hard(P,'low')[0]==ids)
        rnd_ok  = (dec_hard(P,'random',seed=0)[0]==ids)
        TESTS.append((f'soft - hard(low)    {col:<5} {int(round(b["occ"].mean())):>2}c',
                      *paired(soft_ok, low_ok, b['frame'])))
        TESTS.append((f'soft - hard(random) {col:<5} {int(round(b["occ"].mean())):>2}c',
                      *paired(soft_ok, rnd_ok, b['frame'])))

pv = np.array([t[4] for t in TESTS]); sig = holm(pv)
print(f'{len(TESTS)} tests, Holm at alpha 0.05, {sig.sum()} significant\n')
print(f"{'comparison':<30}{'gain':>8}{'95% CI':>20}{'p':>10}  sig")
for (name,obs,lo,hi,p), s in zip(TESTS, sig):
    print(f'{name:<30}{100*obs:>8.2f}  [{100*lo:>6.2f},{100*hi:>6.2f}]{p:>10.4f}  {"yes" if s else "no"}')

gains_low = [100*(r['soft']-r['hard_low'])    for r in rows]
gains_rnd = [100*(r['soft']-r['hard_random']) for r in rows]
print(f'\nmax soft gain over hard with lowest id tie breaking : {max(gains_low):.2f} points')
print(f'max soft gain over hard with random tie breaking    : {max(gains_rnd):.2f} points')
print('\nVERDICT: finding 8 survives only if the random column is still clearly positive.')

36 tests, Holm at alpha 0.05, 35 significant

comparison                        gain              95% CI         p  sig
soft - hard(low)    grey   3c     0.38  [ -0.03,  0.77]    0.0770  no
soft - hard(random) grey   3c    -3.23  [ -3.73, -2.72]    0.0005  yes
soft - hard(low)    grey   6c     1.85  [  1.43,  2.29]    0.0005  yes
soft - hard(random) grey   6c    -3.11  [ -3.63, -2.55]    0.0005  yes
soft - hard(low)    grey   8c     2.42  [  1.98,  2.91]    0.0005  yes
soft - hard(random) grey   8c    -3.90  [ -4.48, -3.30]    0.0005  yes
soft - hard(low)    grey  12c     2.56  [  2.11,  2.98]    0.0005  yes
soft - hard(random) grey  12c    -3.45  [ -3.97, -2.95]    0.0005  yes
soft - hard(low)    grey  16c     4.00  [  3.61,  4.37]    0.0005  yes
soft - hard(random) grey  16c    -2.76  [ -3.26, -2.25]    0.0005  yes
soft - hard(low)    grey  18c     4.16  [  3.76,  4.57]    0.0005  yes
soft - hard(random) grey  18c    -0.82  [ -1.26, -0.39]    0.0020  yes
soft - hard(low)    white  3c

## Why soft decoding helps only under black

The tests above show soft decoding loses to fair hard decoding under grey and white
occlusion, and wins by up to 24 points under black. This cell looks inside the decoder
to see why. For hidden and visible cells separately it reports bit accuracy, mean
confidence, the probability given to the true bit, the share of cells that are wrong
while more than 90% confident, the mean negative log probability of the true bit (the
exact penalty soft decoding pays), and expected calibration error.

If hidden cells under grey and white are confidently wrong while black ones are not,
that is the mechanism behind finding 8.


In [11]:
Pc = decode_published(CLEAN['x']).reshape(len(CLEAN['x']), 36)
ids_c = CLEAN['id']
LPc, LQc = np.log(np.clip(Pc,1e-6,1)), np.log(np.clip(1-Pc,1e-6,1))
rot_score = np.stack([(LPc*ROT[k][ids_c]).sum(1) + (LQc*(1-ROT[k][ids_c])).sum(1) for k in range(4)])
TRUTH = ROT[rot_score.argmax(0), ids_c] > 0.5          # true bits in the decoder's own orientation
clean_acc = ((Pc > 0.5) == TRUTH).mean()
print(f'clean crops: {len(ids_c)} markers, bit accuracy {100*clean_acc:.2f}% (must be near 100)')
assert clean_acc > 0.97, 'orientation recovery failed, do not trust the table below'

def ece(conf, correct, bins=10):
    edges = np.linspace(0.5, 1.0, bins+1)
    idx = np.clip(np.digitize(conf, edges) - 1, 0, bins-1); e = 0.0
    for b in range(bins):
        sel = idx == b
        if sel.any(): e += sel.mean() * abs(correct[sel].mean() - conf[sel].mean())
    return e

CAL = []
print(f"\n{'col':>6}{'cells':>7}{'region':>9}{'bit acc':>9}{'conf':>7}{'p(true)':>9}"
      f"{'conf wrong':>12}{'-log p':>8}{'ECE':>7}")
for col in COLOURS:
    for T in TARGETS:
        b = BUF[(col,T)]
        assert np.array_equal(b['id'], ids_c)
        P = decode_published(b['x']).reshape(-1, 36)
        M = b['mask'].reshape(-1, 36)
        correct = (P > 0.5) == TRUTH
        conf    = np.maximum(P, 1-P)
        p_true  = np.where(TRUTH, P, 1-P)
        for region, sel in (('hidden', M), ('visible', ~M)):
            c, f, pt = correct[sel], conf[sel], p_true[sel]
            r = dict(colour=col, target=T, cells=float(b['occ'].mean()), region=region,
                     n_cells=int(sel.sum()), bit_acc=float(c.mean()), mean_conf=float(f.mean()),
                     p_true=float(pt.mean()), conf_wrong=float(((~c) & (f > 0.9)).mean()),
                     nll=float(-np.log(np.clip(pt,1e-6,1)).mean()), ece=float(ece(f, c)))
            CAL.append(r)
            print(f"{col:>6}{r['cells']:>7.1f}{region:>9}{100*r['bit_acc']:>9.2f}{r['mean_conf']:>7.3f}"
                  f"{r['p_true']:>9.3f}{100*r['conf_wrong']:>11.2f}%{r['nll']:>8.2f}{r['ece']:>7.3f}")
    print()

with open(f'{ART}/n12_cell_calibration_{RUN_ON}.json','w') as fh: json.dump(CAL, fh, indent=1)
print('saved', f'{ART}/n12_cell_calibration_{RUN_ON}.json')


clean crops: 12276 markers, bit accuracy 97.61% (must be near 100)

   col  cells   region  bit acc   conf  p(true)  conf wrong  -log p    ECE
  grey    3.2   hidden    49.48  0.999    0.495      50.41%    6.93  0.505
  grey    3.2  visible    90.01  0.998    0.899       9.73%    1.29  0.098
  grey    5.7   hidden    50.15  0.998    0.501      49.61%    6.80  0.497
  grey    5.7  visible    88.65  0.998    0.886      11.07%    1.47  0.111
  grey    8.5   hidden    53.48  0.999    0.535      46.34%    6.35  0.464
  grey    8.5  visible    87.97  0.998    0.879      11.76%    1.57  0.118
  grey   11.5   hidden    53.95  0.999    0.539      45.82%    6.27  0.459
  grey   11.5  visible    86.55  0.998    0.865      13.13%    1.75  0.132
  grey   15.6   hidden    54.42  0.996    0.544      44.84%    6.15  0.452
  grey   15.6  visible    85.60  0.997    0.856      14.07%    1.86  0.142
  grey   18.0   hidden    53.90  0.996    0.539      45.21%    6.20  0.457
  grey   18.0  visible    83.82 

In [12]:
res = dict(run_on=RUN_ON, position=POSITION, n_draws=N_DRAWS,
           levels={str(k): v for k,v in LEVEL_FOR.items()},
           achieved={str(k): v for k,v in ACHIEVED.items()}, rows=rows,
           tests=[dict(name=t[0], obs=t[1], lo=t[2], hi=t[3], p=t[4], sig=bool(s))
                  for t,s in zip(TESTS, sig)])
with open(f'{ART}/n12_tie_breaking_{RUN_ON}.json','w') as fh: json.dump(res, fh, indent=1)
print('saved', f'{ART}/n12_tie_breaking_{RUN_ON}.json')

saved /content/drive/MyDrive/aruco_p2/artifacts/n12_tie_breaking_dev.json
